# sklearn Linear Regression Baseline


In [1]:
from pathlib import Path

SPLIT_MODE = "drug_blind"  # one of {"drug_blind", "tumor_blind", "mixed"}
SPLIT_FRACTIONS = {"train": 0.8, "val": 0.1, "test": 0.1}
BATCH_SIZE = 512
RANDOM_SEED = 42
NUM_WORKERS = 0

TARGET_MODE = "delta"
PREPROCESS_BATCH_SIZE = 1024
SKLEARN_FIT_INTERCEPT = True
SKLEARN_N_JOBS = -1

PLOT_TEST_SAMPLE_COUNT = 50
PLOT_TEST_SAMPLE_STRATEGY = "seeded_random"
PLOT_RANDOM_SEED = RANDOM_SEED
EMBEDDING_METHOD = "pca"
N_EMBEDDING_COMPONENTS = 2

TENSOR_ARTIFACTS_DIR = Path("data/Tahoe100M_tensor_artifacts_L1000")

print(
    {
        "SPLIT_MODE": SPLIT_MODE,
        "SPLIT_FRACTIONS": SPLIT_FRACTIONS,
        "BATCH_SIZE": BATCH_SIZE,
        "RANDOM_SEED": RANDOM_SEED,
        "NUM_WORKERS": NUM_WORKERS,
        "TARGET_MODE": TARGET_MODE,
        "PREPROCESS_BATCH_SIZE": PREPROCESS_BATCH_SIZE,
        "SKLEARN_FIT_INTERCEPT": SKLEARN_FIT_INTERCEPT,
        "SKLEARN_N_JOBS": SKLEARN_N_JOBS,
        "PLOT_TEST_SAMPLE_COUNT": PLOT_TEST_SAMPLE_COUNT,
        "PLOT_TEST_SAMPLE_STRATEGY": PLOT_TEST_SAMPLE_STRATEGY,
        "PLOT_RANDOM_SEED": PLOT_RANDOM_SEED,
        "EMBEDDING_METHOD": EMBEDDING_METHOD,
        "N_EMBEDDING_COMPONENTS": N_EMBEDDING_COMPONENTS,
        "tumor_blind_behavior": "cell_line_blind",
        "model_family": "sklearn_linear_regression",
    },
)


{'SPLIT_MODE': 'drug_blind', 'SPLIT_FRACTIONS': {'train': 0.8, 'val': 0.1, 'test': 0.1}, 'BATCH_SIZE': 512, 'RANDOM_SEED': 42, 'NUM_WORKERS': 0, 'TARGET_MODE': 'delta', 'PREPROCESS_BATCH_SIZE': 1024, 'SKLEARN_FIT_INTERCEPT': True, 'SKLEARN_N_JOBS': -1, 'PLOT_TEST_SAMPLE_COUNT': 50, 'PLOT_TEST_SAMPLE_STRATEGY': 'seeded_random', 'PLOT_RANDOM_SEED': 42, 'EMBEDDING_METHOD': 'pca', 'N_EMBEDDING_COMPONENTS': 2, 'tumor_blind_behavior': 'cell_line_blind', 'model_family': 'sklearn_linear_regression'}


## Imports and Helpers


In [2]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sklearn.linear_model import LinearRegression
from torch.utils.data import DataLoader

for candidate_root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate_root / "pyproject.toml").exists() and (candidate_root / "Machine_Learning" / "ml_pipeline").exists():
        candidate_root_str = str(candidate_root)
        if candidate_root_str not in sys.path:
            sys.path.insert(0, candidate_root_str)
        break
else:
    raise ModuleNotFoundError(
        "Could not resolve the project root needed to import Machine_Learning.ml_pipeline."
    )

from Machine_Learning.ml_pipeline.data import (
    PreparedTreatmentDataset,
    TrainingPreprocessor,
    TreatmentExampleDataset,
    materialize_prepared_dataset,
)
from Machine_Learning.ml_pipeline.utils import (
    SPLIT_NAMES,
    assign_group_blind_splits,
    assign_mixed_split,
    build_overlap_diagnostics,
    build_prediction_pair_embedding_from_arrays,
    build_project_path,
    build_split_summary,
    evaluate_predictions_from_arrays,
    load_cached_cell_line_metadata,
    resolve_project_path,
    resolve_target_gene_indices,
    sample_prediction_details,
    validate_plot_config,
    validate_preprocessing_config,
    validate_split_assignments,
    validate_split_config,
)


/Users/aniruddh/miniforge3/envs/DeepLearningClass/lib/python3.12/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/Users/aniruddh/miniforge3/envs/DeepLearningClass/lib/python3.12/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <367D4265-B20F-34BD-94EB-4F3EE47C385B> /Users/aniruddh/miniforge3/envs/DeepLearningClass/lib/python3.12/site-packages/torchvision/image.so
  Reason: tried: '/Users/aniruddh/miniforge3/envs/DeepLearningClass/lib/python3.12/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/aniruddh/miniforge3/envs/DeepLearningClass/lib/python3.12/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/aniruddh/miniforge3/envs/DeepLearningClass/lib/python3.12/lib-dynload/../../libjpeg.9.dylib' (no such file), '/Users/aniruddh/miniforge3/envs/DeepLearningClass/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using

## Load Tensors and Supporting Metadata


In [3]:
validate_split_config(SPLIT_MODE, SPLIT_FRACTIONS)
validate_preprocessing_config(
    TARGET_MODE,
    PREPROCESS_BATCH_SIZE,
)
validate_plot_config(
    PLOT_TEST_SAMPLE_COUNT,
    PLOT_TEST_SAMPLE_STRATEGY,
    EMBEDDING_METHOD,
    N_EMBEDDING_COMPONENTS,
)

tensor_artifacts_dir = resolve_project_path(TENSOR_ARTIFACTS_DIR)
dmso_bundle = torch.load(tensor_artifacts_dir / "dmso_baselines.pt", map_location="cpu")
treatment_bundle = torch.load(tensor_artifacts_dir / "treatment_expressions.pt", map_location="cpu")
fingerprint_bundle = torch.load(tensor_artifacts_dir / "morgan_fingerprints.pt", map_location="cpu")

input_gene_ids = [str(gene_id) for gene_id in dmso_bundle["gene_ids"]]
target_gene_ids = [str(gene_id) for gene_id in treatment_bundle["gene_ids"]]
target_gene_indices = resolve_target_gene_indices(input_gene_ids, target_gene_ids)

treatment_feature_space = str(treatment_bundle.get("feature_space", "unknown"))
requested_landmark_gene_ids = [
    str(gene_id) for gene_id in treatment_bundle.get("requested_landmark_gene_ids", target_gene_ids)
]
available_landmark_gene_ids = [
    str(gene_id) for gene_id in treatment_bundle.get("available_landmark_gene_ids", target_gene_ids)
]
missing_landmark_gene_ids = [
    str(gene_id) for gene_id in treatment_bundle.get("missing_landmark_gene_ids", [])
]

if available_landmark_gene_ids and target_gene_ids != available_landmark_gene_ids:
    raise ValueError("Treatment target gene_ids do not match available_landmark_gene_ids metadata.")
if len(target_gene_indices) != len(target_gene_ids):
    raise ValueError("Resolved target gene indices do not match the saved treatment target width.")

cell_line_metadata_df, cell_line_metadata_arrow_path = load_cached_cell_line_metadata()
tensor_cell_lines = set(dmso_bundle["cell_lines"])
matched_cell_line_metadata_df = cell_line_metadata_df.loc[
    cell_line_metadata_df["cell_line"].isin(tensor_cell_lines)
].copy()
missing_cell_line_metadata = sorted(tensor_cell_lines - set(matched_cell_line_metadata_df["cell_line"]))
if missing_cell_line_metadata:
    raise ValueError(f"Missing cell-line metadata for: {missing_cell_line_metadata}")

if matched_cell_line_metadata_df["cell_line"].duplicated().any():
    raise ValueError("cell_line metadata must be unique after deduplication.")

examples_df = pd.DataFrame(
    {
        "condition_key": treatment_bundle["condition_keys"],
        "cell_line": treatment_bundle["cell_lines"],
        "file_name": treatment_bundle["file_names"],
        "drug": treatment_bundle["drug_names"],
        "concentration": treatment_bundle["concentrations"].cpu().numpy().astype(np.float32),
        "concentration_unit": treatment_bundle["concentration_units"],
        "target_index": np.arange(len(treatment_bundle["condition_keys"]), dtype=np.int64),
    }
)
examples_df["baseline_index"] = examples_df["cell_line"].map(dmso_bundle["cell_line_to_index"])
examples_df["fingerprint_index"] = examples_df["drug"].map(fingerprint_bundle["drug_to_index"])

if examples_df["condition_key"].duplicated().any():
    raise ValueError("condition_key values must be unique in the treatment bundle.")
if examples_df["baseline_index"].isna().any():
    raise ValueError("Some treatment rows do not resolve to a DMSO baseline index.")
if examples_df["fingerprint_index"].isna().any():
    raise ValueError("Some treatment rows do not resolve to a Morgan fingerprint index.")

examples_df = examples_df.merge(
    matched_cell_line_metadata_df,
    on="cell_line",
    how="left",
    validate="many_to_one",
)
if examples_df[["cell_name", "organ"]].isna().any().any():
    raise ValueError("Some treatment rows do not resolve to cell-line metadata.")

examples_df[["baseline_index", "fingerprint_index", "target_index"]] = examples_df[[
    "baseline_index",
    "fingerprint_index",
    "target_index",
]].astype(int)

tensor_summary_df = pd.DataFrame(
    [
        {
            "bundle": "dmso_baselines",
            "rows": int(dmso_bundle["expressions"].shape[0]),
            "cols": int(dmso_bundle["expressions"].shape[1]),
            "feature_space": "full_protein_coding",
        },
        {
            "bundle": "treatment_expressions",
            "rows": int(treatment_bundle["expressions"].shape[0]),
            "cols": int(treatment_bundle["expressions"].shape[1]),
            "feature_space": treatment_feature_space,
        },
        {
            "bundle": "morgan_fingerprints",
            "rows": int(fingerprint_bundle["fingerprints"].shape[0]),
            "cols": int(fingerprint_bundle["fingerprints"].shape[1]),
            "feature_space": "morgan_radius2_bits2048",
        },
    ]
)
target_space_summary_df = pd.DataFrame(
    [
        {"metric": "input_gene_count", "value": len(input_gene_ids)},
        {"metric": "target_gene_count", "value": len(target_gene_ids)},
        {"metric": "requested_landmark_gene_count", "value": len(requested_landmark_gene_ids)},
        {"metric": "available_landmark_gene_count", "value": len(available_landmark_gene_ids)},
        {"metric": "missing_landmark_gene_count", "value": len(missing_landmark_gene_ids)},
        {"metric": "treatment_feature_space", "value": treatment_feature_space},
    ]
)
metadata_summary_df = pd.DataFrame(
    [
        {
            "tensor_artifacts_dir": str(tensor_artifacts_dir),
            "cell_line_metadata_arrow_path": str(cell_line_metadata_arrow_path),
            "n_examples": int(len(examples_df)),
            "n_unique_drugs": int(examples_df["drug"].nunique()),
            "n_unique_cell_lines": int(examples_df["cell_line"].nunique()),
            "n_unique_organs": int(examples_df["organ"].nunique()),
        }
    ]
)
missing_landmark_summary_df = pd.DataFrame({"missing_landmark_gene_id": missing_landmark_gene_ids})

print(
    f"Loaded {len(examples_df)} treatment examples with {len(input_gene_ids)} input genes and {len(target_gene_ids)} target genes in {treatment_feature_space}."
)
display(tensor_summary_df)
display(target_space_summary_df)
display(metadata_summary_df)
if not missing_landmark_summary_df.empty:
    display(missing_landmark_summary_df)
display(examples_df.head())


Loaded 26696 treatment examples with 20061 input genes and 977 target genes in l1000_landmark_intersection.


/var/folders/rm/tp3kb8dj25dflt0f18q1f1_h0000gn/T/ipykernel_64980/2474124943.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dmso_bundle = torch.load(tensor_artifacts_di

,bundle,rows,cols,feature_space
0,dmso_baselines,24,20061,full_protein_coding
1,treatment_expressions,26696,977,l1000_landmark_intersection
2,morgan_fingerprints,377,2048,morgan_radius2_bits2048


,metric,value
0,input_gene_count,20061
1,target_gene_count,977
2,requested_landmark_gene_count,978
3,available_landmark_gene_count,977
4,missing_landmark_gene_count,1
5,treatment_feature_space,l1000_landmark_intersection


,tensor_artifacts_dir,cell_line_metadata_arrow_path,n_examples,n_unique_drugs,n_unique_cell_lines,n_unique_organs
0,/Users/aniruddh/Library/CloudStorage/OneDrive-...,/Users/aniruddh/.cache/huggingface/datasets/ve...,26696,377,24,10


,missing_landmark_gene_id
0,ENSG00000153113


,condition_key,cell_line,file_name,drug,concentration,concentration_unit,target_index,baseline_index,fingerprint_index,cell_name,organ
0,CVCL_0023|||(R)-Verapamil (hydrochloride)|||0....,CVCL_0023,CVCL_0023.h5ad,(R)-Verapamil (hydrochloride),0.05,uM,0,0,118,A549,Lung
1,CVCL_0023|||(R)-Verapamil (hydrochloride)|||0....,CVCL_0023,CVCL_0023.h5ad,(R)-Verapamil (hydrochloride),0.50,uM,1,0,118,A549,Lung
2,CVCL_0023|||(R)-Verapamil (hydrochloride)|||5|...,CVCL_0023,CVCL_0023.h5ad,(R)-Verapamil (hydrochloride),5.00,uM,2,0,118,A549,Lung
3,CVCL_0023|||(S)-Crizotinib|||0.05|||uM,CVCL_0023,CVCL_0023.h5ad,(S)-Crizotinib,0.05,uM,3,0,137,A549,Lung
4,CVCL_0023|||(S)-Crizotinib|||0.5|||uM,CVCL_0023,CVCL_0023.h5ad,(S)-Crizotinib,0.50,uM,4,0,137,A549,Lung


## Split Examples and Build DataLoaders


In [4]:
if SPLIT_MODE == "drug_blind":
    split_mode_note = "drug_blind: entire drugs are held out from train."
    split_unit_column = "drug"
    split_assignments = assign_group_blind_splits(
        examples_df,
        group_col="drug",
        split_fractions=SPLIT_FRACTIONS,
        seed=RANDOM_SEED,
    )
elif SPLIT_MODE == "tumor_blind":
    split_mode_note = "tumor_blind: this notebook implements cell-line-blind splits rather than Organ-level splits."
    split_unit_column = "cell_line"
    split_assignments = assign_group_blind_splits(
        examples_df,
        group_col="cell_line",
        split_fractions=SPLIT_FRACTIONS,
        seed=RANDOM_SEED,
    )
else:
    split_mode_note = "mixed: condition keys are held out, but every drug and cell line remains represented in train."
    split_unit_column = "condition_key"
    split_assignments = assign_mixed_split(
        examples_df,
        split_fractions=SPLIT_FRACTIONS,
        seed=RANDOM_SEED,
    )

split_examples_df = examples_df.copy()
split_examples_df["split"] = split_assignments.to_numpy()
validate_split_assignments(split_examples_df, SPLIT_MODE)

split_summary_df = build_split_summary(split_examples_df, SPLIT_FRACTIONS)
split_unit_summary_df = (
    split_examples_df.groupby("split")[split_unit_column]
    .nunique()
    .reindex(SPLIT_NAMES)
    .reset_index(name=f"unique_{split_unit_column}_count")
)
overlap_diagnostics_df = build_overlap_diagnostics(split_examples_df, SPLIT_MODE)

train_examples_df = split_examples_df.loc[split_examples_df["split"] == "train"].reset_index(drop=True)
val_examples_df = split_examples_df.loc[split_examples_df["split"] == "val"].reset_index(drop=True)
test_examples_df = split_examples_df.loc[split_examples_df["split"] == "test"].reset_index(drop=True)
split_sample_counts = {split_name: int((split_examples_df["split"] == split_name).sum()) for split_name in SPLIT_NAMES}

train_dataset = TreatmentExampleDataset(train_examples_df, dmso_bundle, treatment_bundle, fingerprint_bundle)
val_dataset = TreatmentExampleDataset(val_examples_df, dmso_bundle, treatment_bundle, fingerprint_bundle)
test_dataset = TreatmentExampleDataset(test_examples_df, dmso_bundle, treatment_bundle, fingerprint_bundle)

train_generator = torch.Generator().manual_seed(RANDOM_SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    drop_last=False,
    generator=train_generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    drop_last=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    drop_last=False,
)

print(
    f"Using {SPLIT_MODE} split. {split_mode_note} Sample counts -> train: {split_sample_counts['train']}, val: {split_sample_counts['val']}, test: {split_sample_counts['test']}."
)
display(split_summary_df)
display(split_unit_summary_df)
display(overlap_diagnostics_df)


Using drug_blind split. drug_blind: entire drugs are held out from train. Sample counts -> train: 21368, val: 2664, test: 2664.


,split,target_rows,row_count,requested_fraction,realized_fraction,unique_drugs,unique_cell_lines,unique_organs
0,train,21356,21368,0.8,0.80042,303,24,10
1,val,2670,2664,0.1,0.09979,37,24,10
2,test,2670,2664,0.1,0.09979,37,24,10


,split,unique_drug_count
0,train,303
1,val,37
2,test,37


,entity,pair,overlap_count
0,drug,train/val,0
1,drug,train/test,0
2,drug,val/test,0
3,condition_key,train/val,0
4,condition_key,train/test,0
5,condition_key,val/test,0
6,cell_line,train/val,24
7,cell_line,train/test,24
8,cell_line,val/test,24


## Inspect One Batch


In [5]:
batch_examples = {
    "train": next(iter(train_loader)),
    "val": next(iter(val_loader)),
    "test": next(iter(test_loader)),
}

batch_summary_rows = []
for split_name, batch in batch_examples.items():
    split_source_df = split_examples_df.loc[split_examples_df["split"] == split_name].set_index("condition_key")
    first_condition_key = batch["condition_key"][0]
    source_row = split_source_df.loc[first_condition_key]

    if batch["drug"][0] != source_row["drug"] or batch["cell_line"][0] != source_row["cell_line"]:
        raise ValueError("Batch metadata does not align with the split source table.")

    if batch["baseline_expression"].shape[0] > BATCH_SIZE:
        raise ValueError("A batch exceeded the configured batch size.")
    if batch["baseline_expression"].shape[1] != len(dmso_bundle["gene_ids"]):
        raise ValueError("Baseline expression width does not match the gene space.")
    if batch["drug_fingerprint"].shape[1] != fingerprint_bundle["fingerprints"].shape[1]:
        raise ValueError("Drug fingerprint width does not match the saved Morgan tensor.")
    if batch["target_expression"].shape[1] != len(treatment_bundle["gene_ids"]):
        raise ValueError("Target expression width does not match the gene space.")

    batch_summary_rows.append(
        {
            "split": split_name,
            "batch_rows": int(batch["baseline_expression"].shape[0]),
            "baseline_shape": tuple(batch["baseline_expression"].shape),
            "fingerprint_shape": tuple(batch["drug_fingerprint"].shape),
            "concentration_shape": tuple(batch["concentration"].shape),
            "target_shape": tuple(batch["target_expression"].shape),
            "first_condition_key": first_condition_key,
            "first_cell_line": batch["cell_line"][0],
            "first_drug": batch["drug"][0],
        }
    )

batch_summary_df = pd.DataFrame(batch_summary_rows)
display(batch_summary_df)


,split,batch_rows,baseline_shape,fingerprint_shape,concentration_shape,target_shape,first_condition_key,first_cell_line,first_drug
0,train,512,"(512, 20061)","(512, 2048)","(512,)","(512, 977)",CVCL_0359|||Canagliflozin|||0.05|||uM,CVCL_0359,Canagliflozin
1,val,512,"(512, 20061)","(512, 2048)","(512,)","(512, 977)",CVCL_0023|||Adenine|||0.05|||uM,CVCL_0023,Adenine
2,test,512,"(512, 20061)","(512, 2048)","(512,)","(512, 977)",CVCL_0023|||(S)-Crizotinib|||0.05|||uM,CVCL_0023,(S)-Crizotinib


## Fit the Training Preprocessor


In [6]:
training_preprocessor = TrainingPreprocessor(
    train_examples_df=train_examples_df,
    dmso_bundle=dmso_bundle,
    treatment_bundle=treatment_bundle,
    fingerprint_bundle=fingerprint_bundle,
    target_gene_indices=target_gene_indices,
    target_mode=TARGET_MODE,
    batch_size=PREPROCESS_BATCH_SIZE,
).fit()

preprocessor_summary_df = training_preprocessor.summary_frame()
display(preprocessor_summary_df)


,target_mode,train_examples,input_dim,input_gene_dim,target_gene_dim,fingerprint_dim,dose_log_mean,dose_log_std,baseline_std_min,delta_std_min
0,delta,21368,22110,20061,977,2048,-0.307488,0.816949,0.001,0.198753


## Build sklearn Design Matrices and Model


In [7]:
prepared_train_dataset = PreparedTreatmentDataset(train_examples_df, training_preprocessor)
prepared_val_dataset = PreparedTreatmentDataset(val_examples_df, training_preprocessor)
prepared_test_dataset = PreparedTreatmentDataset(test_examples_df, training_preprocessor)

prepared_arrays_by_split = {
    "train": materialize_prepared_dataset(prepared_train_dataset, as_numpy=True),
    "val": materialize_prepared_dataset(prepared_val_dataset, as_numpy=True),
    "test": materialize_prepared_dataset(prepared_test_dataset, as_numpy=True),
}
metadata_by_split = {
    split_name: prepared_arrays_by_split[split_name]["metadata_df"].copy()
    for split_name in SPLIT_NAMES
}
X_by_split = {
    split_name: prepared_arrays_by_split[split_name]["input_features"].astype(np.float32, copy=False)
    for split_name in SPLIT_NAMES
}
y_by_split = {
    split_name: prepared_arrays_by_split[split_name]["target_delta"].astype(np.float32, copy=False)
    for split_name in SPLIT_NAMES
}
baseline_expression_by_split = {
    split_name: prepared_arrays_by_split[split_name]["baseline_expression"].astype(np.float32, copy=False)
    for split_name in SPLIT_NAMES
}
target_expression_by_split = {
    split_name: prepared_arrays_by_split[split_name]["target_expression"].astype(np.float32, copy=False)
    for split_name in SPLIT_NAMES
}

X_train = X_by_split["train"]
y_train = y_by_split["train"]
X_val = X_by_split["val"]
y_val = y_by_split["val"]
X_test = X_by_split["test"]
y_test = y_by_split["test"]

input_feature_dim = int(X_train.shape[1])
target_gene_dim = int(y_train.shape[1])

if input_feature_dim != training_preprocessor.input_dim:
    raise ValueError("Prepared sklearn design matrix width does not match the preprocessor input dimension.")
if target_gene_dim != training_preprocessor.target_gene_dim:
    raise ValueError("Prepared sklearn target width does not match the target gene dimension.")
if baseline_expression_by_split["train"].shape != y_train.shape:
    raise ValueError("Baseline expression and target delta must align in the target gene space.")

prepared_array_summary_df = pd.DataFrame(
    [
        {
            "split": split_name,
            "input_shape": tuple(X_by_split[split_name].shape),
            "baseline_shape": tuple(baseline_expression_by_split[split_name].shape),
            "target_delta_shape": tuple(y_by_split[split_name].shape),
            "target_expression_shape": tuple(target_expression_by_split[split_name].shape),
            "metadata_rows": int(len(metadata_by_split[split_name])),
        }
        for split_name in SPLIT_NAMES
    ]
)

sklearn_model = LinearRegression(
    fit_intercept=SKLEARN_FIT_INTERCEPT,
    n_jobs=SKLEARN_N_JOBS,
)

sklearn_setup_summary_df = pd.DataFrame(
    [
        {
            "model_family": "sklearn_linear_regression",
            "input_feature_dim": input_feature_dim,
            "input_gene_dim": int(training_preprocessor.input_gene_dim),
            "fingerprint_dim": int(training_preprocessor.fingerprint_dim),
            "target_gene_dim": target_gene_dim,
            "fit_intercept": bool(SKLEARN_FIT_INTERCEPT),
            "n_jobs": int(SKLEARN_N_JOBS),
            "treatment_feature_space": treatment_feature_space,
        }
    ]
)

display(sklearn_setup_summary_df)
display(prepared_array_summary_df)


,model_family,input_feature_dim,input_gene_dim,fingerprint_dim,target_gene_dim,fit_intercept,n_jobs,treatment_feature_space
0,sklearn_linear_regression,22110,20061,2048,977,True,-1,l1000_landmark_intersection


,split,input_shape,baseline_shape,target_delta_shape,target_expression_shape,metadata_rows
0,train,"(21368, 22110)","(21368, 977)","(21368, 977)","(21368, 977)",21368
1,val,"(2664, 22110)","(2664, 977)","(2664, 977)","(2664, 977)",2664
2,test,"(2664, 22110)","(2664, 977)","(2664, 977)","(2664, 977)",2664


## Fit sklearn Linear Regression


In [ ]:
sklearn_model.fit(X_train, y_train)

predicted_delta_by_split = {
    split_name: sklearn_model.predict(X_by_split[split_name]).astype(np.float32, copy=False)
    for split_name in SPLIT_NAMES
}
predicted_expression_by_split = {
    split_name: baseline_expression_by_split[split_name] + predicted_delta_by_split[split_name]
    for split_name in SPLIT_NAMES
}

coef_shape = tuple(sklearn_model.coef_.shape)
expected_coef_shape = (target_gene_dim, input_feature_dim)
if coef_shape != expected_coef_shape:
    raise ValueError(f"Unexpected sklearn coefficient shape: {coef_shape} != {expected_coef_shape}")

run_name = f"{SPLIT_MODE}_sklearn_linear_regression_{TARGET_MODE}"
training_run_summary_df = pd.DataFrame(
    [
        {
            "run_name": run_name,
            "model_family": "sklearn_linear_regression",
            "n_features_in": int(sklearn_model.n_features_in_),
            "coef_shape": coef_shape,
            "intercept_shape": tuple(np.asarray(sklearn_model.intercept_).shape),
            "fit_intercept": bool(SKLEARN_FIT_INTERCEPT),
            "n_jobs": int(SKLEARN_N_JOBS),
            "train_rows": int(X_train.shape[0]),
            "val_rows": int(X_val.shape[0]),
            "test_rows": int(X_test.shape[0]),
        }
    ]
)
coefficient_summary_df = pd.DataFrame(
    [
        {
            "coef_mean_abs": float(np.mean(np.abs(sklearn_model.coef_))),
            "coef_max_abs": float(np.max(np.abs(sklearn_model.coef_))),
            "intercept_mean_abs": float(np.mean(np.abs(np.asarray(sklearn_model.intercept_)))),
            "intercept_max_abs": float(np.max(np.abs(np.asarray(sklearn_model.intercept_)))),
        }
    ]
)

display(training_run_summary_df)
display(coefficient_summary_df)


In [ ]:
sklearn_prediction_shape_df = pd.DataFrame(
    [
        {
            "split": split_name,
            "predicted_delta_shape": tuple(predicted_delta_by_split[split_name].shape),
            "predicted_expression_shape": tuple(predicted_expression_by_split[split_name].shape),
        }
        for split_name in SPLIT_NAMES
    ]
)

display(sklearn_prediction_shape_df)


## Evaluate the Fitted sklearn Model


In [ ]:
evaluation_rows = []
inspection_tables = {}
prediction_detail_tables = {}

for split_name in SPLIT_NAMES:
    metrics_row, inspection_df, prediction_details_df = evaluate_predictions_from_arrays(
        predicted_delta=predicted_delta_by_split[split_name],
        baseline_expression=baseline_expression_by_split[split_name],
        target_delta=y_by_split[split_name],
        split_name=split_name,
        gene_ids=target_gene_ids,
        metadata_df=metadata_by_split[split_name],
    )
    evaluation_rows.append(metrics_row)
    inspection_tables[split_name] = inspection_df
    prediction_detail_tables[split_name] = prediction_details_df

evaluation_summary_df = pd.DataFrame(evaluation_rows)
test_prediction_details_df = prediction_detail_tables["test"].copy()
print(f"Evaluated fitted sklearn LinearRegression model for {run_name}")
display(evaluation_summary_df)
display(inspection_tables["test"])
display(test_prediction_details_df.head())


## Plot Train, Validation, and Test Metric Summary


In [ ]:
metric_summary_df = evaluation_summary_df.loc[
    :,
    [
        "split",
        "delta_mse",
        "treated_cosine",
        "mann_whitney_pvalue_mean",
        "signed_ndcg_at_50_mean",
        "top50_deg_match_count_mean",
        "top50_deg_match_fraction_mean",
    ],
].copy().sort_values("split", key=lambda s: s.map({"train": 0, "val": 1, "test": 2}), ignore_index=True)

print("Computed sklearn train/val/test metric summaries from the fitted model.")
display(metric_summary_df)
display(
    evaluation_summary_df[
        [
            "split",
            "top50_deg_match_count_mean",
            "top50_deg_match_count_median",
            "top50_deg_match_fraction_mean",
            "top50_deg_match_fraction_median",
            "signed_ndcg_at_50_mean",
            "signed_ndcg_at_50_median",
        ]
    ]
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
metric_specs = [
    ("delta_mse", "Delta MSE", "#4c72b0"),
    ("treated_cosine", "Treated Cosine", "#55a868"),
    ("mann_whitney_pvalue_mean", "Mean Mann-Whitney P-Value", "#c44e52"),
    ("signed_ndcg_at_50_mean", "Mean Signed nDCG@50", "#8172b2"),
]

for ax, (metric_name, title, color) in zip(axes.flat, metric_specs):
    sns.barplot(
        data=metric_summary_df,
        x="split",
        y=metric_name,
        order=["train", "val", "test"],
        color=color,
        ax=ax,
    )
    if metric_name == "mann_whitney_pvalue_mean":
        ax.axhline(0.05, color="#444444", linestyle="--", linewidth=1.2, label="p = 0.05")
        ax.legend(frameon=False, loc="upper right")
    if metric_name in {"treated_cosine", "signed_ndcg_at_50_mean"}:
        ax.set_ylim(0, 1)
    else:
        ymax = float(metric_summary_df[metric_name].max())
        ax.set_ylim(0, max(ymax * 1.1, 1e-6))
    ax.set_title(f"{title} ({run_name})")
    ax.set_xlabel("Split")
    ax.set_ylabel(title)
    ax.grid(True, axis="y", alpha=0.25)

fig.tight_layout()
plt.show()


## Plot Predicted vs Actual Test Pairs


In [ ]:
sampled_test_prediction_details_df = sample_prediction_details(
    prediction_details_df=test_prediction_details_df,
    sample_count=PLOT_TEST_SAMPLE_COUNT,
    sample_strategy=PLOT_TEST_SAMPLE_STRATEGY,
    random_seed=PLOT_RANDOM_SEED,
)
test_prediction_plot_df, sampled_test_pair_summary_df, test_embedding_explained_variance_ratio = build_prediction_pair_embedding_from_arrays(
    predicted_expression=predicted_expression_by_split["test"],
    actual_expression=target_expression_by_split["test"],
    sampled_prediction_details_df=sampled_test_prediction_details_df,
    embedding_method=EMBEDDING_METHOD,
    n_components=N_EMBEDDING_COMPONENTS,
    random_seed=PLOT_RANDOM_SEED,
)

print(
    f"Plotted {len(sampled_test_prediction_details_df)} {PLOT_TEST_SAMPLE_STRATEGY} test pairs with {EMBEDDING_METHOD.upper()} from the fitted sklearn LinearRegression model."
)
display(sampled_test_pair_summary_df)

fig, ax = plt.subplots(figsize=(12, 9))
for pair_index, pair_points_df in test_prediction_plot_df.groupby("pair_index", sort=True):
    ordered_pair_points_df = pair_points_df.set_index("point_kind").loc[["actual", "predicted"]]
    ax.plot(
        ordered_pair_points_df["embedding_1"],
        ordered_pair_points_df["embedding_2"],
        color="#9e9e9e",
        linewidth=0.8,
        alpha=0.65,
        zorder=1,
    )

sns.scatterplot(
    data=test_prediction_plot_df,
    x="embedding_1",
    y="embedding_2",
    hue="point_kind",
    style="point_kind",
    palette={"actual": "#1f77b4", "predicted": "#d62728"},
    markers={"actual": "o", "predicted": "X"},
    s=90,
    alpha=0.9,
    ax=ax,
)
ax.set_title(
    f"Predicted vs Actual Test Expression in {EMBEDDING_METHOD.upper()} Space ({len(sampled_test_prediction_details_df)} pairs, {run_name})"
)
ax.set_xlabel(
    f"{EMBEDDING_METHOD.upper()} 1 ({test_embedding_explained_variance_ratio[0] * 100:.1f}% var)"
)
ax.set_ylabel(
    f"{EMBEDDING_METHOD.upper()} 2 ({test_embedding_explained_variance_ratio[1] * 100:.1f}% var)"
)
ax.grid(True, alpha=0.25)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
plt.show()


## Plot Test Cell-Line Accuracy


In [ ]:
test_prediction_details_df = test_prediction_details_df.copy()
test_prediction_details_df["is_correct_prediction"] = test_prediction_details_df["mann_whitney_pvalue"] > 0.05

test_cell_line_accuracy_df = (
    test_prediction_details_df.groupby("cell_line", as_index=False)
    .agg(
        n_test_samples=("condition_key", "size"),
        n_correct_predictions=("is_correct_prediction", "sum"),
    )
)
test_cell_line_accuracy_df["n_correct_predictions"] = test_cell_line_accuracy_df["n_correct_predictions"].astype(int)
test_cell_line_accuracy_df["cell_line_accuracy"] = (
    test_cell_line_accuracy_df["n_correct_predictions"] / test_cell_line_accuracy_df["n_test_samples"]
)
test_cell_line_accuracy_df = test_cell_line_accuracy_df.sort_values(
    ["cell_line_accuracy", "cell_line"],
    ascending=[False, True],
    ignore_index=True,
)
mean_cell_line_accuracy = float(test_cell_line_accuracy_df["cell_line_accuracy"].mean())

print("Computed test-set cell-line accuracy using mann_whitney_pvalue > 0.05 as the correctness rule.")
display(test_cell_line_accuracy_df)

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(
    data=test_cell_line_accuracy_df,
    x="cell_line",
    y="cell_line_accuracy",
    color="#4c72b0",
    ax=ax,
)
ax.axhline(
    mean_cell_line_accuracy,
    color="#d62728",
    linestyle="--",
    linewidth=1.5,
    label=f"Mean accuracy = {mean_cell_line_accuracy:.3f}",
)
ax.set_title("Test Cell-Line Accuracy from Mann-Whitney Non-Significance")
ax.set_xlabel("Cell line")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.grid(True, axis="y", alpha=0.25)
ax.tick_params(axis="x", rotation=45)
ax.legend(frameon=False, loc="upper right")
fig.tight_layout()
plt.show()


## Plot Test Cell-Line DEG Overlap Metrics


In [ ]:
test_cell_line_deg_match_df = (
    test_prediction_details_df.groupby("cell_line", as_index=False)
    .agg(
        n_test_samples=("condition_key", "size"),
        mean_top50_deg_match_count=("top50_deg_match_count", "mean"),
        median_top50_deg_match_count=("top50_deg_match_count", "median"),
        min_top50_deg_match_count=("top50_deg_match_count", "min"),
        max_top50_deg_match_count=("top50_deg_match_count", "max"),
    )
    .sort_values(["median_top50_deg_match_count", "cell_line"], ascending=[False, True], ignore_index=True)
)
test_cell_line_signed_ndcg_df = (
    test_prediction_details_df.groupby("cell_line", as_index=False)
    .agg(
        n_test_samples=("condition_key", "size"),
        mean_signed_ndcg_at_50=("signed_ndcg_at_50", "mean"),
        median_signed_ndcg_at_50=("signed_ndcg_at_50", "median"),
        min_signed_ndcg_at_50=("signed_ndcg_at_50", "min"),
        max_signed_ndcg_at_50=("signed_ndcg_at_50", "max"),
    )
    .sort_values(["median_signed_ndcg_at_50", "cell_line"], ascending=[False, True], ignore_index=True)
)
deg_match_order = test_cell_line_deg_match_df["cell_line"].tolist()
signed_ndcg_order = test_cell_line_signed_ndcg_df["cell_line"].tolist()

print("Computed test-set top-50 DEG overlap counts and signed nDCG@50 from predicted vs true delta expression.")
display(test_cell_line_deg_match_df)
display(test_cell_line_signed_ndcg_df)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
deg_ax, ndcg_ax = axes

sns.boxplot(
    data=test_prediction_details_df,
    x="cell_line",
    y="top50_deg_match_count",
    order=deg_match_order,
    whis=(0, 100),
    color="#4c72b0",
    ax=deg_ax,
)
deg_ax.set_title("Test Cell-Line Top-50 DEG Overlap Counts")
deg_ax.set_xlabel("Cell line")
deg_ax.set_ylabel("Matched DEGs out of 50")
deg_ax.grid(True, axis="y", alpha=0.25)
deg_ax.tick_params(axis="x", rotation=45)

sns.boxplot(
    data=test_prediction_details_df,
    x="cell_line",
    y="signed_ndcg_at_50",
    order=signed_ndcg_order,
    whis=(0, 100),
    color="#55a868",
    ax=ndcg_ax,
)
ndcg_ax.set_title("Test Cell-Line Signed nDCG@50")
ndcg_ax.set_xlabel("Cell line")
ndcg_ax.set_ylabel("Signed nDCG@50")
ndcg_ax.set_ylim(0, 1)
ndcg_ax.grid(True, axis="y", alpha=0.25)
ndcg_ax.tick_params(axis="x", rotation=45)

fig.tight_layout()
plt.show()
